<a target="_blank" href="https://colab.research.google.com/github/ddefbcourses/assignment-07-mlp/blob/main/notebooks/assignment.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação baseados em Multi-Layer Perceptrons (MLPs).

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:

- correto
- reproduzível
- rastreável
- criticamente analisado

Além disso, utilizaremos o MLflow para registrar:

- hiperparâmetros
- métricas
- execuções
- comparações
- experimentais

In [1]:
import warnings

warnings.filterwarnings("ignore")

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow

In [5]:
import os

current_dir = os.path.abspath(".")

db_path = os.path.join(current_dir, "..", "mlflow.db").replace("\\", "/")

mlflow.set_tracking_uri(f"sqlite:///{db_path}")

In [6]:
mlflow.set_experiment(
    "assignment"
)

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1779073077774, experiment_id='1', last_update_time=1779073077774, lifecycle_stage='active', name='assignment', tags={}, trace_location=None, workspace='default'>

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST` utilizando fetch_openml.
Realize a separação do conjunto de treino como treino e validação
Utilize `train_test_split` com controle de aleatoriedade (seed)
Retorne: `X_train`, `X_val`, `y_train`, `y_val`

Depois responda:
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [7]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

def load_data(seed):
    dataset = fetch_openml("Fashion-MNIST", version=1, as_frame=False)
    X = dataset.data.astype(np.float32) / 255.0
    y = dataset.target.astype(str)

    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=seed,
        stratify=y,
    )

    return X_train, X_val, y_train, y_val

R: Sim, é recomendado normalizar os dados para MLPs.
Como os pixels do Fashion MNIST variam de 0 a 255, a normalização para a faixa [0, 1] melhora a estabilidade numérica e costuma acelerar a convergência do treinamento.
Sem esse pré-processamento, o gradiente pode ficar menos estável e o modelo tende a treinar de forma mais lenta ou irregular.

# Questão 2

Implemente a função:
`
train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
)
`

## Requisitos:

Utilizar `MLPClassifier` do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [8]:
from sklearn.neural_network import MLPClassifier

def train_mlp(X_train, y_train, activation, hidden_layers, learning_rate, seed):
    
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        learning_rate_init=learning_rate,
        max_iter=200,
        batch_size=32,
        random_state=seed,
        verbose=False
    )
    
    model.fit(X_train, y_train)
    
    return model

# Questão 3

Implemente a função:

`evaluate(model, X_test, y_test)`

Ela deve:

- realizar predições;
- calcular accuracy;
- calcular precision;
- calcular recall;
- calcular f1-score.

**Solução**:

In [9]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(model, X_test, y_test):

    y_pred = model.predict(X_test)
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    metrics = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1
    }
    
    return metrics

# Questão 4

Implemente o rastreamento experimental utilizando MLflow. Devem ser registrados:

Parâmetros
- activation
- hidden_layers
- learning_rate
- max_iter
- batch_size

Métricas
- accuracy
- precision
- recall
- f1_score
- training_time

**Solução**:

In [10]:
def run_experiment(seed, activation, hidden_layers, learning_rate, max_iter=200, batch_size=200):
    
    with mlflow.start_run():
        # Log dos parâmetros
        mlflow.log_param('activation', activation)
        mlflow.log_param('hidden_layers', str(hidden_layers))
        mlflow.log_param('learning_rate', learning_rate)
        mlflow.log_param('max_iter', max_iter)
        mlflow.log_param('batch_size', batch_size)
        
        # Treinar o modelo
        start_time = time.time()
        model = train_mlp(
            X_train, y_train, activation, hidden_layers, learning_rate, seed
        )
        training_time = time.time() - start_time
        
        # Avaliar o modelo
        metrics = evaluate(model, X_val, y_val)
        
        # Log das métricas
        mlflow.log_metric('accuracy', metrics['accuracy'])
        mlflow.log_metric('precision', metrics['precision'])
        mlflow.log_metric('recall', metrics['recall'])
        mlflow.log_metric('f1_score', metrics['f1_score'])
        mlflow.log_metric('training_time', training_time)
        
        print(f'Experimento concluído: activation={activation}, hidden_layers={hidden_layers}')
        print(f'Accuracy: {metrics["accuracy"]:.4f}, F1-score: {metrics["f1_score"]:.4f}, Tempo: {training_time:.2f}s')
        
        return model, metrics, training_time

# Questão 5

Compare diferentes funções de ativação.

- logistic
- tanh
- relu

Você deve registrar todos os experimentos utilizando MLflow.

**Solução**:

In [ ]:
import time

# Configurações fixas para comparação
seed = 42
X_train, X_val, y_train, y_val = load_data(seed)
hidden_layers = (128, 64)
learning_rate = 0.01
activations = ['logistic', 'tanh', 'relu']

results_activation = {}

print('Comparando funções de ativação...')
print('=' * 60)

for activation in activations:
    print(f'\nTestando ativação: {activation}')
    model, metrics, training_time = run_experiment(
        seed=seed,
        activation=activation,
        hidden_layers=hidden_layers,
        learning_rate=learning_rate
    )
    results_activation[activation] = {
        'accuracy': metrics['accuracy'],
        'precision': metrics['precision'],
        'recall': metrics['recall'],
        'f1_score': metrics['f1_score'],
        'training_time': training_time
    }

# Criar DataFrame para análise
df_activations = pd.DataFrame(results_activation).T
print('\n' + '=' * 60)
print('Resumo dos resultados por função de ativação:')
print(df_activations)

# Análise comparativa
print('\n' + '=' * 60)
print('Análise Comparativa:')
print(f'\nMelhor F1-score: {df_activations["f1_score"].idxmax()} ({df_activations["f1_score"].max():.4f})')
print(f'Melhor Accuracy: {df_activations["accuracy"].idxmax()} ({df_activations["accuracy"].max():.4f})')
print(f'Menor tempo de treinamento: {df_activations["training_time"].idxmin()} ({df_activations["training_time"].min():.2f}s)')
print(f'Variabilidade de F1-score (desvio padrão): {df_activations["f1_score"].std():.6f}')

Comparando funções de ativação...

Testando ativação: logistic
Experimento concluído: activation=logistic, hidden_layers=(128, 64)
Accuracy: 0.8382, F1-score: 0.8361, Tempo: 229.74s

Testando ativação: tanh
Experimento concluído: activation=tanh, hidden_layers=(128, 64)
Accuracy: 0.7514, F1-score: 0.7437, Tempo: 144.93s

Testando ativação: relu
Experimento concluído: activation=relu, hidden_layers=(128, 64)
Accuracy: 0.8568, F1-score: 0.8576, Tempo: 1298.14s

Resumo dos resultados por função de ativação:
          accuracy  precision    recall  f1_score  training_time
logistic  0.838214   0.843942  0.838214  0.836062     229.744871
tanh      0.751357   0.748384  0.751357  0.743676     144.929003
relu      0.856786   0.860267  0.856786  0.857606    1298.140852

Análise Comparativa:

Melhor F1-score: relu (0.8576)
Melhor Accuracy: relu (0.8568)
Menor tempo de treinamento: tanh (144.93s)
Variabilidade de F1-score (desvio padrão): 0.060525


## Responda:
- Qual ativação apresentou melhor convergência?
- Qual ativação apresentou maior estabilidade?
- Houve diferenças significativas de treinamento?

## Respostas:

**Qual ativação apresentou melhor convergência?**

A ativação `relu` apresentou a melhor convergência neste experimento, pois obteve o maior `accuracy` (0.8568) e o maior `f1-score` (0.8576). Isso indica melhor capacidade de aprendizado para essa arquitetura e configuração de treino.

**Qual ativação apresentou maior estabilidade?**

Com apenas uma execução por ativação, não é possível medir estabilidade de forma rigorosa. Para isso, seria necessário repetir os experimentos com diferentes seeds e analisar a variância das métricas. Ainda assim, entre os resultados observados, `relu` foi a mais consistente em desempenho, enquanto `tanh` teve os piores valores de accuracy e f1.

**Houve diferenças significativas de treinamento?**

Sim. Houve diferenças claras entre as ativações. `relu` teve o melhor desempenho, mas também foi a mais lenta no treinamento (1298.14s). `tanh` treinou mais rápido (144.93s), porém com desempenho bem inferior. `logistic` ficou em posição intermediária, com accuracy de 0.8382 e f1-score de 0.8361.

# Questão 6

Compare diferentes arquiteturas de MLP.
`
- (32,)
- (64,)
- (128, 64)
- (256, 128)
`

**Solução**:

In [ ]:
# TODO: implemente

## Responda:

- Redes maiores sempre melhoraram os resultados?
- Redes maiores sempre melhoraram os resultados?
- Qual arquitetura apresentou melhor tradeoff?

# Questão 7

Analise o impacto do learning rate.
- 0.1
- 0.01
- 0.001

In [ ]:
# TODO: implemente

## Responda:
- O treinamento ficou instável?
- Houve dificuldade de convergência?
- Qual learning rate apresentou melhor comportamento?

# Questão 8

- Qual ativação apresentou melhor desempenho?
- Qual arquitetura apresentou melhor tradeoff?
- Qual learning rate apresentou maior estabilidade?
- Houve overfitting?
- Qual configuração apresentou melhor resultado final?
- Quais foram as principais dificuldades observadas?


# TODO: responda aqui